# Task 4: Reinforcement Learning from Human Feedback (RLHF)

**Goal:** Fine-tune the pretrained Task 3 Transformer using human preference scores.

**Mathematical Model:**
- Optimization Objective: $\max_\theta J(\theta) = \mathbb{E}[r(X_{gen})]$
- Policy Gradient (REINFORCE): $\nabla_\theta J(\theta) = \mathbb{E}[r \nabla_\theta \log p_\theta(X)]$
- KL Penalty (Guide Fix): $J'(\theta) = \mathbb{E}[r] - \lambda D_{KL}(p_\theta || p_{\theta_0})$ to prevent Reward Hacking.

**Deliverables:** Human survey data integration, reward scoring function, RL tuning loop, 10 final tuned samples.

In [ ]:
import torch, os, math, sys, glob
import numpy as np
import pandas as pd
from torch import nn, optim
import torch.nn.functional as F
import pretty_midi
from miditok import REMI, TokenizerConfig

repo_root = os.path.abspath(os.path.join(os.getcwd(), ".."))
sys.path.append(repo_root)
from generation.midi_export import validate_midi
from evaluation.metrics import evaluate_pair

device = torch.device("cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu")

config = TokenizerConfig(num_velocities=32, use_chords=False, use_programs=False)
tokenizer = REMI(config)
VOCAB_SIZE = tokenizer.vocab_size
PAD_TOKEN = tokenizer['PAD_None']

### 1. Load Pretrained Task 3 Model & Reference Model
We need the active model (`model`) to train, and a frozen reference model (`ref_model`) to compute the KL penalty and prevent the music from devolving into noise just to exploit the reward function.

In [ ]:
processed_tokens_dir = os.path.join("data", "processed", "tokens")
legacy_tokens_dir = os.path.join("data", "processed_tokens")
genre_path = os.path.join(processed_tokens_dir, "genres.npy")
if not os.path.exists(genre_path):
    genre_path = os.path.join(legacy_tokens_dir, "genres.npy")
if os.path.exists(genre_path):
    genre_ids = np.load(genre_path)
    GENRE_COUNT = int(np.max(genre_ids)) + 1 if len(genre_ids) else 1
else:
    GENRE_COUNT = 4

# Re-instantiate the GPT architecture from Task 3
class GPTMusic(nn.Module):
    def __init__(self, vocab_size, genre_count, d_model=256, n_heads=8, num_layers=4):
        super().__init__()
        self.token_emb = nn.Embedding(vocab_size, d_model)
        self.pos_emb = nn.Embedding(1024, d_model)
        self.genre_emb = nn.Embedding(genre_count, d_model)
        layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=n_heads,
            dim_feedforward=d_model * 4,
            batch_first=True,
            dropout=0.1
        )
        self.transformer = nn.TransformerEncoder(layer, num_layers=num_layers)
        self.fc = nn.Linear(d_model, vocab_size)

    def forward(self, x, genre_ids):
        seq_len = x.size(1)
        positions = torch.arange(0, seq_len, device=x.device).unsqueeze(0)
        genre_vec = self.genre_emb(genre_ids).unsqueeze(1)
        x_emb = self.token_emb(x) + self.pos_emb(positions) + genre_vec
        mask = nn.Transformer.generate_square_subsequent_mask(seq_len, device=x.device)
        out = self.transformer(x_emb, mask=mask, is_causal=True)
        return self.fc(out)

model = GPTMusic(VOCAB_SIZE, GENRE_COUNT).to(device)
ref_model = GPTMusic(VOCAB_SIZE, GENRE_COUNT).to(device)

# In practice, load your saved Task 3 weights here:
model_path = 'models/saved/transformer.pth'
if os.path.exists(model_path):
    model.load_state_dict(torch.load(model_path, map_location=device))
    ref_model.load_state_dict(torch.load(model_path, map_location=device))
    print("Loaded pretrained Task 3 weights.")
else:
    print("Warning: Pretrained Task 3 weights not found. Using random init.")

# Freeze the reference model
ref_model.eval()
for param in ref_model.parameters():
    param.requires_grad = False

### 2. Reward Function (Survey-Based)
Load human survey scores from `data/surveys/human_scores.csv`, train a lightweight reward model, and use it to score generated sequences during RLHF.

In [ ]:
survey_path = os.path.join("data", "surveys", "human_scores.csv")
if not os.path.exists(survey_path):
    raise FileNotFoundError("Missing survey CSV at data/surveys/human_scores.csv with columns: midi_path, score")

survey_df = pd.read_csv(survey_path)
required_cols = {"midi_path", "score"}
if not required_cols.issubset(set(survey_df.columns)):
    raise ValueError("survey CSV must include midi_path and score columns")

def midi_features_from_path(midi_path):
    pm = pretty_midi.PrettyMIDI(midi_path)
    notes = [n for inst in pm.instruments for n in inst.notes]
    if not notes:
        return np.zeros(5, dtype=np.float32)
    duration = max(pm.get_end_time(), 1e-3)
    pitches = np.array([n.pitch for n in notes])
    starts = np.array(sorted([n.start for n in notes]))
    gaps = np.diff(starts) if len(starts) > 1 else np.array([duration])
    note_density = len(notes) / duration
    unique_ratio = len(np.unique(pitches)) / max(len(pitches), 1)
    pitch_std = np.std(pitches)
    avg_gap = float(np.mean(gaps))
    return np.array([note_density, unique_ratio, pitch_std, avg_gap, duration], dtype=np.float32)

features = []
scores = []
for _, row in survey_df.iterrows():
    if not os.path.exists(row["midi_path"]):
        continue
    features.append(midi_features_from_path(row["midi_path"]))
    scores.append(float(row["score"]))

if len(features) == 0:
    raise ValueError("No valid midi_path entries found in survey CSV")

X = torch.tensor(np.stack(features), dtype=torch.float32).to(device)
y = torch.tensor(scores, dtype=torch.float32).to(device)

class RewardNet(nn.Module):
    def __init__(self, in_dim=5, hidden=32):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, hidden),
            nn.ReLU(),
            nn.Linear(hidden, 1)
        )
    def forward(self, feats):
        return self.net(feats).squeeze(-1)

reward_model = RewardNet().to(device)
reward_opt = optim.Adam(reward_model.parameters(), lr=1e-3)
reward_loss_fn = nn.MSELoss()

for epoch in range(1, 51):
    reward_model.train()
    reward_opt.zero_grad()
    preds = reward_model(X)
    loss = reward_loss_fn(preds, y)
    loss.backward()
    reward_opt.step()
    if epoch % 10 == 0:
        print(f"RewardNet Epoch {epoch}: MSE {loss.item():.4f}")

def reward_from_tokens(tokens):
    pm = tokenizer.tokens_to_midi(tokens)
    notes = [n for inst in pm.instruments for n in inst.notes]
    if not notes:
        return torch.tensor(0.0, device=device)
    duration = max(pm.get_end_time(), 1e-3)
    pitches = np.array([n.pitch for n in notes])
    starts = np.array(sorted([n.start for n in notes]))
    gaps = np.diff(starts) if len(starts) > 1 else np.array([duration])
    feats = np.array([
        len(notes) / duration,
        len(np.unique(pitches)) / max(len(pitches), 1),
        np.std(pitches),
        float(np.mean(gaps)),
        duration,
    ], dtype=np.float32)
    with torch.no_grad():
        return reward_model(torch.tensor(feats, device=device).unsqueeze(0)).squeeze(0)

### 3. Policy Gradient Update Loop (REINFORCE with Baseline & KL Penalty)

In [ ]:
optimizer = optim.Adam(model.parameters(), lr=5e-5)
KL_BETA = 0.1
RL_STEPS = 30
BATCH_SIZE = 6
SEQ_LEN = 256

def sample_next_token(logits, temperature=1.0, top_k=20):
    logits = logits / max(temperature, 1e-6)
    if top_k is not None and top_k > 0:
        values, indices = torch.topk(logits, top_k)
        probs = torch.softmax(values, dim=-1)
        return indices[torch.multinomial(probs, 1)].item()
    probs = torch.softmax(logits, dim=-1)
    return torch.multinomial(probs, 1).item()

def get_bos_token():
    try:
        return tokenizer['BOS_None']
    except Exception:
        return None

def generate_sequence(model, genre_id, max_len=256, temperature=1.1, top_k=20):
    bos = get_bos_token()
    tokens = [bos] if bos is not None else [np.random.randint(0, VOCAB_SIZE)]
    model.eval()
    for _ in range(max_len - 1):
        x = torch.tensor(tokens, dtype=torch.long, device=device).unsqueeze(0)
        genre = torch.tensor([genre_id], dtype=torch.long, device=device)
        with torch.no_grad():
            logits = model(x, genre)[:, -1, :].squeeze(0)
        tokens.append(sample_next_token(logits, temperature, top_k))
    return tokens

def pad_tokens(token_lists, pad_value, max_len):
    padded = []
    for seq in token_lists:
        seq = seq[:max_len]
        if len(seq) < max_len:
            seq = seq + [pad_value] * (max_len - len(seq))
        padded.append(seq)
    return torch.tensor(padded, dtype=torch.long, device=device)

print("Starting RLHF Tuning...")
model.train()
for step in range(1, RL_STEPS + 1):
    tokens_list = []
    genre_ids = []
    for i in range(BATCH_SIZE):
        gid = i % max(GENRE_COUNT, 1)
        tokens_list.append(generate_sequence(model, gid, max_len=SEQ_LEN))
        genre_ids.append(gid)
    tokens_tensor = pad_tokens(tokens_list, PAD_TOKEN, SEQ_LEN)
    genre_tensor = torch.tensor(genre_ids, dtype=torch.long, device=device)
    optimizer.zero_grad()
    logits = model(tokens_tensor[:, :-1], genre_tensor)
    log_probs = F.log_softmax(logits, dim=-1)
    targets = tokens_tensor[:, 1:].unsqueeze(-1)
    selected_log_probs = log_probs.gather(2, targets).squeeze(-1).sum(dim=1)
    with torch.no_grad():
        ref_logits = ref_model(tokens_tensor[:, :-1], genre_tensor)
        ref_log_probs = F.log_softmax(ref_logits, dim=-1)
        ref_selected_log_probs = ref_log_probs.gather(2, targets).squeeze(-1).sum(dim=1)
    kl_div = selected_log_probs - ref_selected_log_probs
    rewards = torch.stack([reward_from_tokens(toks) for toks in tokens_list]).to(device)
    normalized_rewards = (rewards - rewards.mean()) / (rewards.std() + 1e-8)
    rl_loss = - (normalized_rewards * selected_log_probs).mean()
    kl_loss = KL_BETA * kl_div.mean()
    total_loss = rl_loss + kl_loss
    total_loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
    optimizer.step()
    if step % 10 == 0:
        print(f"Step {step}/{RL_STEPS} | Total Loss: {total_loss.item():.4f} | RL Obj: {-rl_loss.item():.4f} | KL Pen: {kl_loss.item():.4f}")

os.makedirs('models/saved', exist_ok=True)
torch.save(model.state_dict(), 'models/saved/transformer_rlhf.pth')
print("RLHF Tuning Complete.")

output_dir = os.path.join("outputs", "generated_midis", "task4")
os.makedirs(output_dir, exist_ok=True)
generated_paths = []
for i in range(10):
    gid = i % max(GENRE_COUNT, 1)
    tokens = generate_sequence(model, gid, max_len=512, temperature=1.0, top_k=20)
    pm = tokenizer.tokens_to_midi(tokens)
    out_path = os.path.join(output_dir, f"genre_{gid}_sample_{i+1}.mid")
    pm.write(out_path)
    if validate_midi(out_path):
        generated_paths.append(out_path)
    else:
        os.remove(out_path)

def find_reference_midi():
    candidates = glob.glob(os.path.join("data", "raw_midi", "maestro-v3.0.0", "**", "*.mid"), recursive=True)
    return candidates[0] if candidates else None

def evaluate_folder(folder, ref_path):
    midi_files = sorted(glob.glob(os.path.join(folder, "*.mid")))
    rows = []
    for midi_path in midi_files:
        if ref_path:
            rows.append(evaluate_pair(ref_path, midi_path))
        else:
            rows.append({"pitch_hist": np.nan, "rhythm_diversity": np.nan, "repetition_ratio": np.nan})
    if not rows:
        return None
    return pd.DataFrame(rows).mean().to_dict()

ref_midi = find_reference_midi()
comparison_rows = []
comparison_map = {
    "Random": os.path.join("outputs", "generated_midis", "baseline_random"),
    "Markov": os.path.join("outputs", "generated_midis", "baseline_markov"),
    "Task1_LSTM": os.path.join("outputs", "generated_midis", "task1"),
    "Task2_VAE": os.path.join("outputs", "generated_midis", "task2"),
    "Task3_Transformer": os.path.join("outputs", "generated_midis", "task3"),
    "Task4_RLHF": output_dir,
}
for name, folder in comparison_map.items():
    if os.path.exists(folder):
        metrics = evaluate_folder(folder, ref_midi)
        if metrics:
            comparison_rows.append({"model": name, **metrics})
comparison_df = pd.DataFrame(comparison_rows)
comparison_df

Starting RLHF Tuning...
Step 10/50 | Total Loss: -3.9346 | RL Obj: 4.0089 | KL Pen: 0.0743
Step 20/50 | Total Loss: 0.8061 | RL Obj: -0.6687 | KL Pen: 0.1373
Step 10/50 | Total Loss: -3.9346 | RL Obj: 4.0089 | KL Pen: 0.0743
Step 20/50 | Total Loss: 0.8061 | RL Obj: -0.6687 | KL Pen: 0.1373
Step 30/50 | Total Loss: -0.2354 | RL Obj: -0.2167 | KL Pen: -0.4522
Step 40/50 | Total Loss: 0.7933 | RL Obj: -1.0085 | KL Pen: -0.2153
Step 30/50 | Total Loss: -0.2354 | RL Obj: -0.2167 | KL Pen: -0.4522
Step 40/50 | Total Loss: 0.7933 | RL Obj: -1.0085 | KL Pen: -0.2153
Step 50/50 | Total Loss: -0.9045 | RL Obj: 0.0955 | KL Pen: -0.8090
RLHF Tuning Complete.
Step 50/50 | Total Loss: -0.9045 | RL Obj: 0.0955 | KL Pen: -0.8090
RLHF Tuning Complete.


In [ ]:
from generation.midi_export import validate_midi

base_dir = os.path.join("outputs", "generated_midis")
folders = [
    "baseline_random",
    "baseline_markov",
    "task1",
    "task2",
    "task3",
    "task4",
]

validation_rows = []
for folder in folders:
    full_path = os.path.join(base_dir, folder)
    if not os.path.exists(full_path):
        continue
    midi_files = glob.glob(os.path.join(full_path, "*.mid"))
    valid = 0
    for midi_path in midi_files:
        if validate_midi(midi_path):
            valid += 1
    validation_rows.append({
        "folder": folder,
        "count": len(midi_files),
        "valid": valid,
    })

validation_df = pd.DataFrame(validation_rows)
validation_df